# Make spatial metrics from MNT

In [ ]:
import os
import sys

from matplotlib import colors

os.environ["PROJ_LIB"] = "/home/abernard/miniconda3/envs/phenosensing/share/proj"
os.environ["PROJ_DATA"] = "/home/abernard/miniconda3/envs/phenosensing/share/proj"
os.environ["GDAL_DATA"] = "/home/abernard/miniconda3/envs/phenosensing/share/gdal"


import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import rioxarray as rio
import pandas as pd
import phenosensing
from rasterio.enums import Resampling

from pyproj import CRS

print(CRS.from_epsg(2154))

In [ ]:
import rasterio
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.merge import merge

In [ ]:
# read some other data

gdf = gpd.read_file("../data/RNN_limite_poly_l93.shp")

gdf = gdf.set_crs("EPSG:2154")

gdf = gdf.to_crs("EPSG:4326")

In [ ]:
mnt_dir = Path("../../PhenoAnalysis/data/MNT_lidarHD")

mnt_files = [
    p for p in mnt_dir.rglob("*")
    if p.is_file()
    and "_MNT_" in p.name
    and not p.name.endswith(".json")
]


for mnt in sorted(mnt_files):

    with rasterio.open(mnt) as src:

        print("\n", mnt.name)
        print("  CRS         :", src.crs)
        print("  Résolution  :", src.res)
        print("  Dimensions  :", src.width, "x", src.height)
        print("  Type        :", src.dtypes[0])
        print("  Bounds      :", src.bounds)

In [ ]:
src_files = [
    rasterio.open(mnt)
    for mnt in sorted(mnt_files)
]

print(f"{len(src_files)} tuiles ouvertes")

In [ ]:
# build the mnt (all tiles)

mnt_mosaic, mosaic_transform = merge(
    src_files,
    nodata=-9999
)

# merge() retourne un tableau de forme :
# (band, y, x)
mnt_mosaic = mnt_mosaic[0]

mnt_mosaic = mnt_mosaic.astype("float32")

mnt_mosaic[mnt_mosaic == -9999] = np.nan

print("\nMosaïque créée")
print("Shape :", mnt_mosaic.shape)
print("Nombre de pixels :", mnt_mosaic.size)

In [ ]:
# plot and make sure everything is ok

from rasterio.plot import plotting_extent

fig, ax = plt.subplots(figsize=(12, 10))

# Emprise spatiale réelle du raster
extent = plotting_extent(
    mnt_mosaic,
    mosaic_transform
)

# Afficher le MNT
im = ax.imshow(
    mnt_mosaic,
    cmap="terrain",
    extent=extent
)

# Mettre le polygone dans le même CRS que le MNT
gdf_2154 = gdf.to_crs("EPSG:2154")

# Afficher le contour
gdf_2154.boundary.plot(
    ax=ax,
    color="red",
    linewidth=2
)

# Colorbar
fig.colorbar(
    im,
    ax=ax,
    label="Altitude (m)"
)

ax.set_title("MNT LiDAR HD – mosaïque des 34 tuiles")
ax.set_xlabel("Lambert-93 X (m)")
ax.set_ylabel("Lambert-93 Y (m)")

plt.tight_layout()
plt.show()

In [ ]:
# ecrire le mnt complet dans un fichier
output_mnt = mnt_dir / "mnt_mosaic.tif"

profile = src_files[0].profile.copy()

profile.update(
    height=mnt_mosaic.shape[0],
    width=mnt_mosaic.shape[1],
    transform=mosaic_transform
)

with rasterio.open(output_mnt, "w", **profile) as dst:
    dst.write(np.nan_to_num(mnt_mosaic, nan=-9999), 1)

In [ ]:
# load some libraries to test

In [ ]:
# 1. Configuration des chemins GRASS
grass_bin = "/usr/lib/grass83"
os.environ["GISBASE"] = grass_bin
sys.path.append(os.path.join(grass_bin, "etc", "python"))

# 2. Importer la bibliothèque de scripts GRASS
import grass.script as gs

grassdata = Path("grassdata").resolve()

In [ ]:
# a lancer la toute premiere fois car GRASS version 8.3 (8.4 plus simple)


#
# import os
# import sys
# import tempfile
#
# # 1. Configuration des chemins GRASS
# grass_bin = "/usr/lib/grass83"
# os.environ["GISBASE"] = grass_bin
# sys.path.append(os.path.join(grass_bin, "etc", "python"))
#
# # 2. Importer la bibliothèque de scripts GRASS
# import grass.script as gs
#
# # 3. Création d'une session temporaire "à blanc" (Fichier gisrc factice)
# # Cela empêche g.gisenv de planter lors de l'appel à create_location
# fd, mock_gisrc = tempfile.mkstemp()
# os.close(fd)
# os.environ["GISRC"] = mock_gisrc
#
# # Définir des variables d'environnement minimales attendues par GRASS
# os.environ["GRASS_VERSION"] = "8.3"
# with open(mock_gisrc, "w") as f:
#     f.write("GISDBASE: /tmp\n")
#     f.write("LOCATION_NAME: temp_init\n")
#     f.write("MAPSET: PERMANENT\n")
#
# # 4. Création de votre VRAIE location
# grassdata = Path("grassdata").resolve()
# os.makedirs(grassdata, exist_ok=True)
#
# print("Création de la location 'project_massane'...")
# gs.create_location(
#     dbase=str(grassdata),
#     location="project_massane",
#     epsg=2154
# )
# print("Location créée avec succès !")
#
# # 5. Nettoyage du fichier temporaire initial
# if os.path.exists(mock_gisrc):
#     os.remove(mock_gisrc)


In [ ]:
import os
import sys
import tempfile

# 6. Démarrage de votre vraie session de travail sur le projet tout juste créé
from grass.script import setup as gsetup
gsetup.init(grassdata, "project_massane", "PERMANENT")

print("Session GRASS GIS initialisée sur project_massane [EPSG:2154].")

test de l'exemple : https://github.com/ncsu-geoforall-lab/gis-week-grass-2024/blob/main/02_jupyter_tutorial.ipynb

In [ ]:
# Import GRASS packages
import grass.script as gs
import grass.jupyter as gj

# Start GRASS Session
session = gj.init(grassdata, "project_massane", "PERMANENT")



In [ ]:
# importer le MNT natif

gs.run_command(
    "r.import",
    input=str(output_mnt),
    output="mnt_mosaic"
)

In [ ]:
# Set computational region to the elevation raster.
gs.run_command("g.region",
               raster="mnt_mosaic")

In [ ]:
# degrader à 2m

gs.run_command(
    "g.region",
    raster="mnt_mosaic",
    res=2
)

gs.run_command(
    "r.resamp.stats",
    input="mnt_mosaic",
    output="mnt_2m",
    method="average"
)

In [ ]:
print(gs.parse_command("g.region", flags="p"))

tutorial DEM : https://grass-tutorials.osgeo.org/content/tutorials/terrain_and_DEMs/GRASS_terrain.html

seulement le relief (figuratif, pas de parameter heure)

In [ ]:
gs.run_command("r.relief",input="mnt_2m", output="relief", zscale=3)

In [ ]:
shadow_map = gj.Map()
shadow_map.d_rast(map="relief")
shadow_map.show()

combiner relier et alt (couleur)

In [ ]:
gs.run_command("r.shade",
                shade="relief",
                color="mnt_2m",
                output="topography_colored_relief",
                brighten=40)

In [ ]:
alt_shadow_map = gj.Map(
    width=1200,
    height=900
)
alt_shadow_map.d_rast(map="topography_colored_relief")
alt_shadow_map.show()

## terrain et slope (et hydro)

In [ ]:
gs.run_command("r.slope.aspect",
                elevation="mnt_2m",
                slope="slope",
                aspect="aspect",
                flags="n")

In [ ]:
slope_map = gj.Map(width=1200, height=900)

slope_map.d_rast(
    map="slope"
)

slope_map.d_legend(
    raster="slope",
    title="Slope",
    units="°",
    labelnum=4,
    range=(0, 75),
    fontsize=12,
    color="white"
)

slope_map.show()

In [ ]:
# landforms (28mins traitement à 200m natif, 1min à 200m 2m de res)

gs.run_command("r.geomorphon",
                elevation="mnt_2m",
                forms="landforms200",
                search=200,
                skip=10,
                flat=1,
                comparison="anglev2_distance")

In [ ]:
landform_map = gj.Map(width=1200, height=900)

landform_map.d_rast(
    map="landforms200"
)

landform_map.d_legend(
    raster="landforms200",
    title="Geomorphons",
    fontsize=12,
    color="white"
)

landform_map.show()

hydro

In [ ]:
gs.run_command("r.watershed",
                elevation="mnt_2m",
                threshold=10000,
                accumulation="watershed_accumulation10000",
                basin="watershed_basin10000",
                stream="watershed_streams10000",
                flags="ab")

In [ ]:
watershed_ac_100000 = gj.Map(width=1200, height=900)

watershed_ac_100000.d_rast(
    map="watershed_accumulation10000",
)

watershed_ac_100000.show()

In [ ]:
watershed_bassin_100000 = gj.Map(width=1200, height=900)

watershed_bassin_100000.d_rast(
    map="watershed_basin10000",
)

watershed_bassin_100000.show()

In [ ]:
gs.run_command("r.shade",
                shade="relief",
                color="watershed_basin10000",
                output="bassin_colored_relief",
                brighten=40)

In [ ]:
bassin_colored_relief_map = gj.Map(width=1200, height=900)

bassin_colored_relief_map.d_rast(
    map="bassin_colored_relief",
)

bassin_colored_relief_map.show()

## ensoleillement

In [ ]:
# horizon topo pour accelerer les calculs ensuite (3mins à 2m) (step 30 = 7mins, step 10 = 24mins)

gs.run_command(
    "r.horizon",
    elevation="mnt_2m",
    output="horizon",
    step=10,
    maxdistance=3000,
    bufferzone=200
)

In [ ]:
gs.run_command(
    "r.sun",
    elevation="mnt_2m",
    slope="slope",
    aspect="aspect",
    horizon_basename="horizon",
    horizon_step=10,
    day=250,
    insol_time="sunshine_172",
    glob_rad="global_rad_172",
)

In [ ]:
sun_map = gj.Map(width=1200, height=900)

sun_map.d_rast(map="sunshine_172")

sun_map.d_legend(
    raster="sunshine_172",
    title="Durée d'ensoleillement",
    units="heures",
    fontsize=12,
    color="white"
)

sun_map.show()

In [ ]:
rad_map = gj.Map(width=1200, height=900)

rad_map.d_rast(
    map="global_rad_172"
)

rad_map.d_legend(
    raster="global_rad_172",
    title="Rayonnement solaire global",
    units="Wh/m²",
    fontsize=12,
    color="white"
)

rad_map.d_barscale(
    units="kilometers",
    fontsize=12
)

rad_map.show()

In [ ]:
print(gs.raster_info("global_rad_172"))

In [ ]:
# compute the metrics

In [ ]:
# export the metrics